In [1]:
# import necessary libraries
import numpy as np
import torch
import torch.nn as nn

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from torch.utils.data import TensorDataset, DataLoader

In [2]:
# generate the synthetic dataset

np.random.seed(42)

n_samples = 1000
d = 2

mu_0 = np.array([-2.0, 0.0])
mu_1 = np.array([2.0, 0.0])

sigma = 1.0

y = np.random.randint(0, 2, size=n_samples)

X = np.zeros((n_samples, d))

for i in range(n_samples):
    if y[i] == 0:
        X[i] = np.random.normal(mu_0, sigma, d)
    else:
        X[i] = np.random.normal(mu_1, sigma, d)

In [3]:
# find 10 nearest neighbors for each point in the dataset

k = 10

knn = NearestNeighbors(n_neighbors=k + 1)
knn.fit(X)

distances, indices = knn.kneighbors(X)

neighbor_indices = indices[:, 1:]

In [4]:
# apply neighborhood dependent label modification
neighbor_label_mean = y[neighbor_indices].mean(axis=1)

epsilon = 0.1

ambiguous = np.abs(neighbor_label_mean - 0.5) < epsilon

y_modified = y.copy()

y_modified[ambiguous] = 1 - y_modified[ambiguous]

print("Number of flipped labels:", ambiguous.sum())
print("Percentage flipped:", ambiguous.mean() * 100)

Number of flipped labels: 22
Percentage flipped: 2.1999999999999997


In [5]:
# create center and neighbor arrays
X_center = X
X_neighbors = X[neighbor_indices]

print("Center shape:", X_center.shape)
print("Neighbors shape:", X_neighbors.shape)
print("Labels shape:", y_modified.shape)

Center shape: (1000, 2)
Neighbors shape: (1000, 10, 2)
Labels shape: (1000,)


In [7]:
# train-test split
X_center_train, X_center_test, X_neighbors_train, X_neighbors_test, y_train, y_test = train_test_split(
    X_center,
    X_neighbors,
    y_modified,
    test_size=0.2,
    random_state=42,
    stratify=y_modified
)

# convert to tensors
X_center_train = torch.tensor(X_center_train, dtype=torch.float32)
X_center_test = torch.tensor(X_center_test, dtype=torch.float32)

X_neighbors_train = torch.tensor(X_neighbors_train, dtype=torch.float32)
X_neighbors_test = torch.tensor(X_neighbors_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [8]:
# apply data loaders
train_dataset = TensorDataset(
    X_center_train,
    X_neighbors_train,
    y_train
)

test_dataset = TensorDataset(
    X_center_test,
    X_neighbors_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [12]:
# define base function for Janossy pooling
class JanossyFunction(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(10 * 2 + 2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, center, neighbors):
        neighbors = neighbors.reshape(
            neighbors.size(0), -1
        )

        combined = torch.cat(
            [center, neighbors],
            dim=1
        )

        output = self.network(combined)

        return output.squeeze(1)

In [13]:
# test base function
janossy_function = JanossyFunction()

center_batch = X_center_train[:32]
neighbors_batch = X_neighbors_train[:32]

output = janossy_function(
    center_batch,
    neighbors_batch
)

print("Input center shape:", center_batch.shape)
print("Input neighbors shape:", neighbors_batch.shape)
print("Output shape:", output.shape)

Input center shape: torch.Size([32, 2])
Input neighbors shape: torch.Size([32, 10, 2])
Output shape: torch.Size([32])


In [14]:
# define Janossy pooling model
class JanossyPooling(nn.Module):

    def __init__(self, num_permutations=10):
        super().__init__()

        self.num_permutations = num_permutations
        self.function = JanossyFunction()

    def forward(self, center, neighbors):

        outputs = []

        for _ in range(self.num_permutations):

            permutation = torch.randperm(
                neighbors.size(1),
                device=neighbors.device
            )

            shuffled_neighbors = neighbors[:, permutation, :]

            output = self.function(
                center,
                shuffled_neighbors
            )

            outputs.append(output)

        outputs = torch.stack(outputs, dim=1)

        averaged_output = outputs.mean(dim=1)

        return averaged_output

In [15]:
# instantiate and test the Janossy pooling model
model = JanossyPooling(
    num_permutations=10
)

output = model(
    center_batch,
    neighbors_batch
)

print("Output shape:", output.shape)

Output shape: torch.Size([32])


In [16]:
# set loss and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [17]:
# train the Janossy model
num_epochs = 50

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for center_batch, neighbors_batch, labels_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(
            center_batch,
            neighbors_batch
        )

        loss = criterion(
            outputs,
            labels_batch
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}: "
            f"Loss = {average_loss:.4f}"
        )

Epoch 10: Loss = 0.0650
Epoch 20: Loss = 0.0602
Epoch 30: Loss = 0.0557
Epoch 40: Loss = 0.0523
Epoch 50: Loss = 0.0492


In [18]:
# evaluate Janossy pooling
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for center_batch, neighbors_batch, labels_batch in test_loader:

        outputs = model(
            center_batch,
            neighbors_batch
        )

        probabilities = torch.sigmoid(outputs)

        predictions = (
            probabilities >= 0.5
        ).float()

        all_predictions.extend(
            predictions.numpy()
        )

        all_labels.extend(
            labels_batch.numpy()
        )

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Class 0", "Class 1"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)

Accuracy: 0.9650

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.96      0.97      0.96        98
     Class 1       0.97      0.96      0.97       102

    accuracy                           0.96       200
   macro avg       0.96      0.97      0.96       200
weighted avg       0.97      0.96      0.97       200

Confusion Matrix:
[[95  3]
 [ 4 98]]


In [19]:
# Janossy permutation test
model.eval()

center = X_center_test[0:1]
neighbors = X_neighbors_test[0:1]

with torch.no_grad():

    original_probability = torch.sigmoid(
        model(center, neighbors)
    ).item()

differences = []

for _ in range(10):

    permutation = torch.randperm(
        neighbors.shape[1]
    )

    shuffled_neighbors = neighbors[:, permutation, :]

    with torch.no_grad():

        shuffled_probability = torch.sigmoid(
            model(center, shuffled_neighbors)
        ).item()

    differences.append(
        abs(
            original_probability
            - shuffled_probability
        )
    )

print(
    "Original probability:",
    original_probability
)

print(
    "Maximum difference:",
    max(differences)
)

print(
    "All differences:",
    differences
)

Original probability: 0.9985412359237671
Maximum difference: 0.00013065338134765625
All differences: [8.940696716308594e-06, 1.2218952178955078e-05, 2.9742717742919922e-05, 6.431341171264648e-05, 3.218650817871094e-05, 0.00013065338134765625, 3.445148468017578e-05, 2.86102294921875e-06, 8.368492126464844e-05, 5.328655242919922e-05]


In [20]:
# create fixed permutations
torch.manual_seed(42)

num_permutations = 10
num_neighbors = 10

fixed_permutations = []

for _ in range(num_permutations):
    fixed_permutations.append(
        torch.randperm(num_neighbors)
    )

fixed_permutations = torch.stack(
    fixed_permutations
)

print("Fixed permutations shape:", fixed_permutations.shape)

Fixed permutations shape: torch.Size([10, 10])


In [21]:
# create deterministic Janossy pooling model
class JanossyPoolingFixed(nn.Module):

    def __init__(self, janossy_function, permutations):
        super().__init__()

        self.function = janossy_function
        self.permutations = permutations

    def forward(self, center, neighbors):

        outputs = []

        for permutation in self.permutations:

            shuffled_neighbors = neighbors[:, permutation, :]

            output = self.function(
                center,
                shuffled_neighbors
            )

            outputs.append(output)

        outputs = torch.stack(
            outputs,
            dim=1
        )

        averaged_output = outputs.mean(dim=1)

        return averaged_output

In [23]:
# instantiate the Janossy pooling model

fixed_model = JanossyPoolingFixed(
    janossy_function=model.function,
    permutations=fixed_permutations
)

In [24]:
# evaluate the Janossy model
fixed_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for center_batch, neighbors_batch, labels_batch in test_loader:

        outputs = fixed_model(
            center_batch,
            neighbors_batch
        )

        probabilities = torch.sigmoid(outputs)

        predictions = (
            probabilities >= 0.5
        ).float()

        all_predictions.extend(
            predictions.numpy()
        )

        all_labels.extend(
            labels_batch.numpy()
        )

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Class 0", "Class 1"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)

Accuracy: 0.9650

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.96      0.97      0.96        98
     Class 1       0.97      0.96      0.97       102

    accuracy                           0.96       200
   macro avg       0.96      0.97      0.96       200
weighted avg       0.97      0.96      0.97       200

Confusion Matrix:
[[95  3]
 [ 4 98]]
